# NaNLM r32 — 30-minute hierarchical multidomain run

This notebook trains a **151M-total / ~21M-active** domain→branch→pair MoE on the 16 public NaNROUTER `corpora/domains/` releases.

- Data builder samples 4 MB from each of 16 domains.
- Domain token `257+d` conditions every window.
- Router: 8 branches × 16 pairs; pick 1 branch, top-2 pairs inside it.
- Training is wall-clock capped at **30 minutes** (`timeout 1800`); the last periodic checkpoint is probed per domain.

Runtime: ~2–3 min data + 30 min train + ~2–5 min probe. A free T4 is enough; an A100/H100 does more positions in the same 30 minutes.


In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


## 1. Clone the repo

Everything needed is in the repo; no private Kaggle dataset is required.


In [ ]:
%cd /content
!rm -rf nanlm_experimental
!git clone --depth 1 https://github.com/shemshallah/nanlm_experimental
%cd nanlm_experimental
!git log --oneline -3


## 2. Build the 16-domain corpus

Downloads ~64 MB of public GitHub release text via HTTP range requests, then writes `train.bin` / `valid.bin` / `meta.json`.


In [ ]:
!python r32/code/prepare_domains.py \
  --out-dir /content/r32data --cache-dir /content/r32cache \
  --per-domain-mb 4 --chunks 4 --valid-fraction 0.10 --seed 2026


In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


## 3. Train for 30 minutes

`timeout 1800` stops the process at 30 minutes. The trainer saves every 100 steps, so the last checkpoint is at most 100 steps behind. If you hit CUDA OOM, change `--batch 32` to `--batch 16` (and expect fewer positions).

The 151M additive MoE must use the default `grouped` dispatch on T4; the `gather` path OOMs.


**Progress logging:** the trainer prints at step 1 and then every `eval-every` steps. This notebook uses `--eval-every 25`, so you should see a line roughly every 50–90 seconds on a T4. If you see no line for several minutes, check `nvidia-smi`; the 30-minute `timeout` will still stop and checkpoint the run.


In [ ]:
!bash -c 'timeout 1800 python -u r32/code/poincare_frontier.py train \
  --data-dir /content/r32data --run-dir /content/run_hier \
  --steps 100000 --context 257 --train-context 256 --eval-context 256 \
  --batch 32 --grad-accum 1 \
  --future2 0.3 --future3 0.15 --future-mode rollout \
  --lr 3e-4 --eval-every 25 --eval-batches 1 --save-every 50 --seed 1337 \
  --dim 544 --layers 4 --heads 16 --pairs 128 --topk 2 \
  --expert-hidden 235 --moe-every 2 \
  --geo 1.0 --balance 0.01 --router-mode both --moe-additive \
  --hierarchical --branches 8 --pairs-per-branch 16 2>&1 | tee /content/train_hier.log || true'
echo "training window ended; last checkpoint:"
!ls -lh /content/run_hier/latest.pt


## 4. Per-domain probe

65k+ held-out positions per run: 16 domains × 4 batches × batch 8 × 256. Reports BPB/top-1 per domain, 128-pair routing, branch routing, domain-token norms, geometry.


In [ ]:
!python -u r32/code/probe_r32.py \
  --data-dir /content/r32data --run-dir /content/run_hier \
  --context 256 --batch 8 --batches 4 \
  --out /content/hier.probe.json


## 5. Summarize

In [ ]:
import json, pandas as pd
p = json.load(open('/content/hier.probe.json'))
print('aggregate:', p['aggregate'])
print('checkpoint step:', p['checkpoint_step'], 'params:', p['parameters'])
rows = []
for d in p['per_domain']:
    if 'error' in d:
        rows.append({'domain': d['domain'], 'error': d['error']}); continue
    u = d['union']; b = d.get('branch_union', {})
    rows.append({'domain': d['domain'], 'bpb': d['bpb'], 'top1': d['top1'],
                 'active_pairs': u['active_pairs'], 'eff_pairs': u['effective_pairs'],
                 'branch_eff': b.get('effective_pairs'),
                 'dom_token_norm': d['domain_token_norm']})
df = pd.DataFrame(rows).sort_values('bpb')
display(df)


## 6. Download results

The probe JSON and training log are small; the checkpoint is large. Download the JSON/log if you want me to analyze or log the run in the repo.


In [ ]:
!cd /content && zip -q -r r32_results.zip hier.probe.json train_hier.log 2>/dev/null || true
from google.colab import files
files.download('/content/r32_results.zip')


## Colab limits and GPU notes

- Colab free has **no fixed weekly GPU-hour allowance**. GPU access is dynamic and not guaranteed; a free notebook can run for at most 12 hours and often ends sooner. You may get a T4; a particular GPU is not guaranteed.
- Colab Pro is 100 compute units/month (roughly 55–85 T4-hours or ~18 A100-hours at measured 2026 rates). Pro+ is 500–600 units/month with priority A100/H100 and background execution up to 24h.
- There is **no T100 GPU** in Colab or Kaggle. Colab offers T4/L4/V100/A100/G4/H100 depending on plan and availability. Kaggle offers T4×2 or P100, with a **30 GPU-hours/week** quota.
- Colab Enterprise (GCP) is pay-as-you-go: roughly $0.42/hr list for T4, $4.71/hr for A100 40GB, $11.75/hr for H100, with spot/committed discounts; billing must be enabled.
